# LLM → SPICE → GDS (AI Agentic Layout)
SSCS Chipathon 2026 — gLayout Track

Describe your analog circuit in natural language.
An LLM (DeepSeek) generates the SPICE netlist,
then gLayout produces the DRC-clean GDSII layout.

**PDK**: gf180mcuD  |  **Supply**: 1.8 V  |  **LLM**: DeepSeek

In [ ]:
import sys, os
from pathlib import Path
root = Path("/foss/designs/chipathon2026-D")
sys.path.insert(0, str(root))

from glayout import gf180
from gdsfactory import Component
import gdstk, IPython.display

from core import (
    llm_to_gds, generate_netlist_from_prompt,
    spice_to_gds, display_component,
    run_ota_ac, run_drc, run_lvs, run_pex,
)

In [ ]:
# ==========================================
# 0. Setup API Key
# ==========================================
# Option A: export DEEPSEEK_API_KEY=sk-...
# Option B: edit file .env (copy from .env.example)

# Verify key is loaded:
from core.pipeline import _load_api_key
key = _load_api_key()
if key:
    print(f"API key loaded: {key[:10]}...")
else:
    print("WARNING: DEEPSEEK_API_KEY not set!")
    print("  Run: export DEEPSEEK_API_KEY=sk-...")
    print("  Or:  cp .env.example .env && edit .env")

In [ ]:
# ==========================================
# 1. Describe Your Circuit (Natural Language)
# ==========================================

prompt = """
Design a 5-transistor operational transconductance amplifier (OTA)
with differential NMOS input pair, PMOS current mirror load,
and NMOS tail current source.
Use gf180mcuD PDK with 1.8V supply.
Input pair: W=10u L=1u, current mirror: W=20u L=1u, tail: W=15u L=1u.
Subcircuit name: my_ota
Ports: vin_p, vin_n, vout, vbias, vdd, vss
"""

In [ ]:
# ==========================================
# 2. LLM → SPICE Netlist
# ==========================================

netlist = generate_netlist_from_prompt(prompt)
if netlist:
    print(netlist)
else:
    print("LLM failed to generate netlist. Check your API key.")

In [ ]:
# ==========================================
# 3. SPICE → GDS Layout
# ==========================================

if netlist:
    GDS_PATH = os.path.join(os.getcwd(), "llm_out.gds")
    result = spice_to_gds(netlist, mode="analog", add_labels=True)
    result.write_gds(GDS_PATH)
    print(f"Layout written to {GDS_PATH}")
    display_component(result, scale=2)

In [ ]:
# ==========================================
# 4. One-Liner: LLM → GDS (end-to-end)
# ==========================================
# Single call — LLM generates netlist, gLayout produces GDS:

# result = llm_to_gds(
#     "Design a 5T OTA with diff pair, PMOS load, NMOS tail. 1.8V supply."
# )
# result.write_gds("my_circuit.gds")